# Phase 05 · Bài 06 — Named Entity Recognition

**Mục tiêu:** hiểu NER như bài gán nhãn chuỗi; tự cài BIO/BILOU và round-trip span; đo bằng entity-level F1 thay vì token accuracy; thấy vì sao ràng buộc chuyển trạng thái của CRF quan trọng; nhận diện giới hạn nested entity.

## 1. Bài toán

`Apple sued Google over its iPhone search deal in the US.` chứa năm thực thể: Apple (ORG), Google (ORG), iPhone (PRODUCT), `search deal` (có thể), US (GPE). Hệ NER tốt trích xuất đủ và đúng kiểu. Hệ NER tệ bỏ sót iPhone, nhầm Apple quả với Apple công ty, hoặc gán US là PERSON.

NER là bộ phận chìm dưới mọi pipeline trích xuất có cấu trúc: parse CV, quét log tuân thủ, ẩn danh hồ sơ y tế, hiểu truy vấn tìm kiếm, grounding câu trả lời chatbot, bóc hợp đồng pháp lý. Người dùng hiếm khi thấy nó, nhưng luôn phụ thuộc vào nó.

## 2. BIO và BILOU

Gán nhãn BIO biến trích xuất thực thể thành gán nhãn chuỗi:

```text
Apple    B-ORG
sued     O
Google   B-ORG
over     O
its      O
iPhone   B-PRODUCT
search   O
deal     O
in       O
the      O
US       B-GPE
.        O
```

`B-X` mở đầu một thực thể kiểu X, `I-X` nằm bên trong, `O` nằm ngoài. Thực thể nhiều token nối tiếp: `New/B-GPE York/I-GPE City/I-GPE`.

BILOU tách thêm `L-X` (token cuối) và `U-X` (thực thể một token). Với BILOU, ranh giới rõ hơn: một chuỗi `B-X` không có `L-X` là dấu hiệu lỗi, trong khi BIO không phân biệt được "kết thúc đúng" và "bị cắt cụt". Đổi lại BILOU có nhiều nhãn hơn, dữ liệu nhãn hiếm hơn.

Một điểm quan trọng: BIO cho phép các chuỗi *bất hợp lệ* như `O I-ORG` hoặc `B-GPE I-ORG`. Bộ giải mã chỉ lấy argmax từng token sẽ tạo ra chúng thường xuyên. CRF học được rằng các chuyển tiếp đó gần như không xảy ra, nên đường đi tối ưu toàn cục luôn hợp lệ.

## 3. Tiến trình kiến trúc

- **Rule-based + gazetteer.** Regex và từ điển. Precision cao với thực thể đã biết, coverage bằng 0 với thực thể mới. Không phân biệt được `Apple` công ty và quả táo.
- **HMM.** Xác suất phát xạ P(token | tag) và chuyển tiếp P(tag_i | tag_{i-1}); giải mã bằng Viterbi. Học từ dữ liệu gán nhãn, nhưng đặc trưng chỉ là chính token.
- **CRF.** Mô hình phân biệt, cho phép đặc trưng tùy ý: hình dạng từ, tiền tố/hậu tố, từ trước/sau, chữ hoa. Vẫn là workhorse cổ điển cho triển khai ít tài nguyên.
- **BiLSTM-CRF.** Thay đặc trưng thủ công bằng đặc trưng học được; CRF trên cùng vẫn ép chuỗi nhãn hợp lệ.
- **Transformer fine-tune.** BERT + token-classification head. Accuracy cao nhất, chi phí lớn nhất.
- **LLM.** Zero-shot / few-shot, cạnh tranh khi thiếu dữ liệu gán nhãn; yếu ở nhất quán định dạng và chi phí.

## 4. Đặc trưng thủ công

Với NER cổ điển, đặc trưng là cuộc chơi. Những đặc trưng hữu ích:

- chữ thường của token,
- `isupper`, `istitle`, `isdigit`,
- hậu tố 3 ký tự,
- word shape: `iPhone → xXxxxx`, `USA-2024 → XXX-dddd`,
- token trước và sau,
- vị trí đầu/cuối câu.

Chữ hoa là tín hiệu mạnh cho danh từ riêng trong tiếng Anh, nhưng không đáng tin trong tiếng Đức (mọi danh từ đều viết hoa) hoặc văn bản viết thường toàn bộ.

## 5. Đo lường: entity-level, không phải token-level

Token accuracy dễ bị đánh lừa vì phần lớn token là `O`. Một mô hình gán mọi token thành `O` có thể đạt 85–90% token accuracy trên corpus thưa thực thể, trong khi entity F1 bằng 0.

Entity-level F1 yêu cầu span dự đoán khớp *chính xác* span vàng: cùng start, cùng end, cùng kiểu. Đây là metric đúng. Thư viện chuẩn là `seqeval`.

Với thực thể nhiều token, sai một token cũng làm hỏng cả span. Vì vậy token accuracy 0.95 có thể đi kèm entity F1 0.6. Luôn báo cả hai, và luôn báo per-type.

## 6. Nested entity và các giới hạn

- **Nested:** `Bank of America Tower` vừa là ORG vừa là FACILITY. BIO chuẩn không biểu diễn được hai span chồng lấn. Cần multi-pass (mỗi kiểu một lượt) hoặc mô hình span-based.
- **Domain shift:** NER huấn luyện trên CoNLL kém hơn gazetteer trên hợp đồng pháp lý. Phải fine-tune theo miền.
- **Thực thể dài:** `United States Federal Deposit Insurance Corporation` dễ bị cắt; cần aggregation hoặc post-process.
- **Kiểu hiếm:** DRUG_BRAND, ADVERSE_EVENT, DOSE trong y tế. Mô hình tổng quát không biết.

## 7. LLM cho NER (2026)

- **Zero-shot:** đưa danh sách kiểu thực thể + schema JSON, yêu cầu JSON. Chạy được ngay, accuracy trung bình trên miền mới.
- **ZeroTuneBio-style:** tách thành trích candidate → giải thích nghĩa → phán quyết → kiểm tra lại. Nhiều bước tốt hơn one-shot trên NER y sinh.
- **Dynamic prompting + RAG:** truy hồi các ví dụ gán nhãn gần nhất cho từng lần suy luận, dựng few-shot prompt động.
- **Per-entity-type decomposition:** tài liệu dài, một lượt trích tất cả kiểu sẽ giảm recall khi độ dài tăng. Chạy một lượt cho mỗi kiểu.

Khuyến nghị: bắt đầu bằng LLM zero-shot trước khi thu thập dữ liệu gán nhãn; thường F1 đủ dùng.

## 8. Khi nào NER cổ điển vẫn thắng

- latency dưới 50 ms,
- có hàng nghìn ví dụ gán nhãn và cần F1 trên 98%,
- ontology ổn định, CRF/BiLSTM pretrained transfer tốt,
- yêu cầu on-prem, mô hình không sinh văn bản.


## A. Chạy code mẫu của bài

Cell dưới chạy `code/main.py` của bài 06: gán nhãn câu `Apple sued Google over iPhone sales in the US .` bằng gazetteer, đổi BIO sang span, in word shape, và kiểm tra round-trip `spans → BIO → spans` trên câu có thực thể nhiều token.


In [1]:
from pathlib import Path
import subprocess
import sys

PHASE = "phases/05-nlp-foundations-to-advanced"
LESSON = "06-named-entity-recognition"
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / PHASE / "README.md").is_file():
        REPO = candidate
        break
else:
    raise FileNotFoundError("Mở Jupyter từ thư mục repo hoặc thư mục con của repo.")
script = REPO / PHASE / LESSON / "code" / "main.py"
result = subprocess.run([sys.executable, str(script)], cwd=REPO, text=True, capture_output=True)
print("Bài:", "Named Entity Recognition")
print("Lệnh:", script.relative_to(REPO))
print("Python:", sys.version.split()[0])
print("Exit code:", result.returncode)
print("\n--- stdout ---")
print(result.stdout.rstrip())
if result.stderr:
    print("\n--- stderr ---")
    print(result.stderr.rstrip())
if result.returncode != 0:
    raise RuntimeError(f"Code mẫu của {LESSON} thoát với mã {result.returncode}")


Bài: Named Entity Recognition
Lệnh: phases\05-nlp-foundations-to-advanced\06-named-entity-recognition\code\main.py
Python: 3.12.10
Exit code: 0

--- stdout ---
tokens  : ['Apple', 'sued', 'Google', 'over', 'iPhone', 'sales', 'in', 'the', 'US', '.']
labels  : ['B-ORG', 'O', 'B-ORG', 'O', 'B-PRODUCT', 'O', 'O', 'O', 'B-GPE', 'O']
spans   :
  [0:1] ORG      'Apple'
  [2:3] ORG      'Google'
  [4:5] PRODUCT  'iPhone'
  [8:9] GPE      'US'

word shapes (useful CRF features):
  Apple        -> shape Xxxxx
  iPhone       -> shape xXxxxx
  IBM          -> shape XXX
  USA-2024     -> shape XXX-dddd
  apple        -> shape xxxxx

round-trip (spans -> BIO -> spans):
  tokens   : ['The', 'New', 'York', 'City', 'mayor', 'visited', 'OpenAI', '.']
  bio      : ['O', 'B-GPE', 'I-GPE', 'I-GPE', 'O', 'O', 'B-ORG', 'O']
  gold     : [(1, 4, 'GPE'), (6, 7, 'ORG')]
  recovered: [(1, 4, 'GPE'), (6, 7, 'ORG')]
  match    : True


## B. Thực hành riêng — đo entity-level F1 và sửa chuỗi bất hợp lệ

Hai việc mà code mẫu chưa làm.

**Đo đúng metric.** Ta tạo một tập test có thực thể ngoài gazetteer (`Tesla`, `Berlin`, `Satya Nadella`) và một bộ gán nhãn theo luật: khớp cụm trong gazetteer trước, sau đó đoán `B-ORG` cho token viết hoa chưa biết. Bộ luật này sai ở đâu đó, và ta so **token accuracy** với **entity-level F1** để thấy token accuracy che lỗi ranh giới/kiểu như thế nào.

**Giải mã có ràng buộc.** Ta cho mỗi token một điểm phát xạ (giả lập đầu ra mô hình), lấy argmax từng token, rồi kiểm tra chuỗi nhãn có hợp lệ theo BIO không. Sau đó chạy Viterbi với transition chỉ cho phép `O/B-X → O/B-X`, `B-X → I-X`, `I-X → I-X`, và `I-X` không được nhảy sang kiểu khác. So sánh cho thấy ràng buộc sửa được chuỗi mà argmax làm hỏng.

Ghi chú: điểm phát xạ ở đây do người viết đặt để cô lập hành vi giải mã; đây không phải mô hình đã huấn luyện.


In [2]:
import math
from collections import Counter

LABELS = ["O", "B-ORG", "I-ORG", "B-GPE", "I-GPE", "B-PERSON", "I-PERSON", "B-PRODUCT", "I-PRODUCT"]

PHRASE_GAZETTEER = {
    ("New", "York", "City"): "GPE",
    ("Bank", "of", "America"): "ORG",
}
TOKEN_GAZETTEER = {
    "Apple": "ORG",
    "Google": "ORG",
    "Microsoft": "ORG",
    "OpenAI": "ORG",
    "Meta": "ORG",
    "Amazon": "ORG",
    "Netflix": "ORG",
    "Anthropic": "ORG",
    "US": "GPE",
    "USA": "GPE",
    "UK": "GPE",
    "India": "GPE",
    "Germany": "GPE",
    "France": "GPE",
    "Japan": "GPE",
    "iPhone": "PRODUCT",
    "Android": "PRODUCT",
    "Windows": "PRODUCT",
    "ChatGPT": "PRODUCT",
    "Claude": "PRODUCT",
    "Gemini": "PRODUCT",
}
SENTENCE_STARTERS = {"The", "A", "An", "In", "On", "At", "This", "It", "We", "I", "For", "And", "But"}


def spans_to_bio(n_tokens, spans):
    labels = ["O"] * n_tokens
    for start, end, kind in spans:
        labels[start] = "B-" + kind
        for i in range(start + 1, end):
            labels[i] = "I-" + kind
    return labels


def bio_to_spans(labels):
    spans, current = [], None
    for i, label in enumerate(labels):
        if label.startswith("B-"):
            if current:
                spans.append(current)
            current = (i, i + 1, label[2:])
        elif label.startswith("I-") and current and current[2] == label[2:]:
            current = (current[0], i + 1, current[2])
        else:
            if current:
                spans.append(current)
                current = None
    if current:
        spans.append(current)
    return spans


def rule_based_ner(tokens):
    labels = ["O"] * len(tokens)
    i = 0
    while i < len(tokens):
        matched = None
        for phrase, kind in PHRASE_GAZETTEER.items():
            if tuple(tokens[i:i + len(phrase)]) == phrase:
                matched = (phrase, kind)
                break
        if matched:
            phrase, kind = matched
            labels[i] = "B-" + kind
            for offset in range(1, len(phrase)):
                labels[i + offset] = "I-" + kind
            i += len(phrase)
            continue
        token = tokens[i]
        if token in TOKEN_GAZETTEER:
            labels[i] = "B-" + TOKEN_GAZETTEER[token]
        elif token.istitle() and token not in SENTENCE_STARTERS:
            labels[i] = "B-ORG"
            if i + 1 < len(tokens) and tokens[i + 1].istitle() and tokens[i + 1] not in TOKEN_GAZETTEER:
                labels[i + 1] = "I-ORG"
                i += 1
        i += 1
    return labels


TEST = [
    (["Tesla", "opened", "a", "plant", "in", "Berlin", "."],
     [(0, 1, "ORG"), (5, 6, "GPE")]),
    (["The", "New", "York", "City", "mayor", "met", "Satya", "Nadella", "."],
     [(1, 4, "GPE"), (6, 8, "PERSON")]),
    (["Anthropic", "released", "Claude", "for", "Android", "in", "Japan", "."],
     [(0, 1, "ORG"), (2, 3, "PRODUCT"), (4, 5, "PRODUCT"), (6, 7, "GPE")]),
]


def entity_scores(gold_spans, pred_spans, kinds=("ORG", "GPE", "PERSON", "PRODUCT")):
    gold = Counter(gold_spans)
    pred = Counter(pred_spans)
    rows, macro = {}, 0.0
    for kind in kinds:
        g = Counter({span: n for span, n in gold.items() if span[2] == kind})
        p = Counter({span: n for span, n in pred.items() if span[2] == kind})
        tp = sum((g & p).values())
        fp = sum((p - g).values())
        fn = sum((g - p).values())
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows[kind] = (tp, fp, fn, precision, recall, f1)
        macro += f1
    tp_all = sum(rows[k][0] for k in kinds)
    fp_all = sum(rows[k][1] for k in kinds)
    fn_all = sum(rows[k][2] for k in kinds)
    micro_p = tp_all / (tp_all + fp_all) if tp_all + fp_all else 0.0
    micro_r = tp_all / (tp_all + fn_all) if tp_all + fn_all else 0.0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if micro_p + micro_r else 0.0
    return rows, micro_p, micro_r, micro_f1, macro / len(kinds)


print("=== rule-based NER: token accuracy vs entity F1 ===")
gold_all, pred_all = [], []
correct_tokens = total_tokens = 0
for tokens, gold_spans in TEST:
    gold_labels = spans_to_bio(len(tokens), gold_spans)
    pred_labels = rule_based_ner(tokens)
    pred_spans = bio_to_spans(pred_labels)
    correct_tokens += sum(1 for g, p in zip(gold_labels, pred_labels) if g == p)
    total_tokens += len(tokens)
    gold_all.extend(gold_spans)
    pred_all.extend(pred_spans)
    print(f"\n  {' '.join(tokens)}")
    print(f"    gold: {gold_spans}")
    print(f"    pred: {pred_spans}")

rows, micro_p, micro_r, micro_f1, macro_f1 = entity_scores(gold_all, pred_all)
print(f"\n  token accuracy = {correct_tokens / total_tokens:.3f}")
print(f"  entity precision = {micro_p:.3f}  recall = {micro_r:.3f}  micro-F1 = {micro_f1:.3f}  macro-F1 = {macro_f1:.3f}")
print(f"  {'kiểu':10s} {'TP':>3s} {'FP':>3s} {'FN':>3s} {'P':>6s} {'R':>6s} {'F1':>6s}")
for kind, (tp, fp, fn, p, r, f1) in rows.items():
    print(f"  {kind:10s} {tp:3d} {fp:3d} {fn:3d} {p:6.3f} {r:6.3f} {f1:6.3f}")

print("\n=== giải mã có ràng buộc BIO ===")
DECODE_LABELS = ["O", "B-GPE", "I-GPE", "B-ORG", "I-ORG"]
tokens = ["The", "New", "York", "City", "mayor"]
emissions = [
    {"O": 4.0, "B-GPE": 0.1, "I-GPE": 0.1, "B-ORG": 0.1, "I-ORG": 0.1},
    {"O": 3.2, "B-GPE": 3.0, "I-GPE": 0.2, "B-ORG": 1.0, "I-ORG": 0.1},
    {"O": 0.2, "B-GPE": 2.6, "I-GPE": 3.4, "B-ORG": 1.2, "I-ORG": 0.5},
    {"O": 0.7, "B-GPE": 2.4, "I-GPE": 3.1, "B-ORG": 0.9, "I-ORG": 0.4},
    {"O": 4.0, "B-GPE": 0.1, "I-GPE": 0.1, "B-ORG": 0.1, "I-ORG": 0.1},
]


def legal_transition(previous, current):
    if current.startswith("I-"):
        return previous.endswith(current[1:]) and previous[0] in "BI"
    return True


def viterbi_decode(emissions, labels, constrained=True):
    n = len(emissions)
    scores = [{label: emissions[0][label] for label in labels}]
    back = [{label: None for label in labels}]
    for i in range(1, n):
        scores.append({})
        back.append({})
        for label in labels:
            best_score, best_prev = -math.inf, None
            for previous in labels:
                if constrained and not legal_transition(previous, label):
                    continue
                score = scores[i - 1][previous] + emissions[i][label]
                if score > best_score:
                    best_score, best_prev = score, previous
            scores[i][label] = best_score
            back[i][label] = best_prev
    last = max(labels, key=lambda label: scores[-1][label])
    path = [last]
    for i in range(n - 1, 0, -1):
        path.append(back[i][path[-1]])
    return list(reversed(path))


def illegal_transitions(path):
    return [(i, path[i - 1], path[i]) for i in range(1, len(path)) if not legal_transition(path[i - 1], path[i])]


argmax_path = [max(DECODE_LABELS, key=lambda label: row[label]) for row in emissions]
free_path = viterbi_decode(emissions, DECODE_LABELS, constrained=False)
constrained_path = viterbi_decode(emissions, DECODE_LABELS, constrained=True)

print("tokens          :", tokens)
print("per-token argmax:", argmax_path)
print("viterbi không ràng buộc:", free_path, "(trùng argmax vì transition = 0)")
print("viterbi có ràng buộc  :", constrained_path)
print("vi phạm của argmax    :", illegal_transitions(argmax_path))
print("vi phạm của constrained:", illegal_transitions(constrained_path))
print("span từ argmax        :", bio_to_spans(argmax_path))
print("span từ constrained   :", bio_to_spans(constrained_path))
print("Gold                  : [(1, 4, 'GPE')]")


=== rule-based NER: token accuracy vs entity F1 ===

  Tesla opened a plant in Berlin .
    gold: [(0, 1, 'ORG'), (5, 6, 'GPE')]
    pred: [(0, 1, 'ORG'), (5, 6, 'ORG')]

  The New York City mayor met Satya Nadella .
    gold: [(1, 4, 'GPE'), (6, 8, 'PERSON')]
    pred: [(1, 4, 'GPE'), (6, 8, 'ORG')]

  Anthropic released Claude for Android in Japan .
    gold: [(0, 1, 'ORG'), (2, 3, 'PRODUCT'), (4, 5, 'PRODUCT'), (6, 7, 'GPE')]
    pred: [(0, 1, 'ORG'), (2, 3, 'PRODUCT'), (4, 5, 'PRODUCT'), (6, 7, 'GPE')]

  token accuracy = 0.875
  entity precision = 0.750  recall = 0.750  micro-F1 = 0.750  macro-F1 = 0.617
  kiểu        TP  FP  FN      P      R     F1
  ORG          2   2   0  0.500  1.000  0.667
  GPE          2   0   1  1.000  0.667  0.800
  PERSON       0   0   1  0.000  0.000  0.000
  PRODUCT      2   0   0  1.000  1.000  1.000

=== giải mã có ràng buộc BIO ===
tokens          : ['The', 'New', 'York', 'City', 'mayor']
per-token argmax: ['O', 'O', 'I-GPE', 'I-GPE', 'O']
viterbi k

## C. Thí nghiệm — nested entity và BILOU

BIO chuẩn giả định mỗi token thuộc tối đa một thực thể. `Bank of America Tower` phá giả định đó: cả `Bank of America` (ORG) và `Bank of America Tower` (FACILITY) đều đúng. Ghi lần lượt hai span vào cùng mảng nhãn thì span sau ghi đè span trước; thông tin mất, không phải chỉ bị nén.

Cách xử lý phổ biến: **multi-pass theo kiểu**. Mỗi kiểu thực thể có một lượt gán nhãn nhị phân riêng; kết quả là nhiều mảng nhãn chồng lên nhau. Đây là cách các pipeline trích xuất hợp đồng/y tế thường làm khi ontology có thực thể lồng nhau.

Phần thứ hai so BILOU với BIO trên thực thể một token và nhiều token, để thấy `L-`/`U-` mã hóa ranh giới rõ hơn.


In [3]:
def spans_to_bio(n_tokens, spans):
    labels = ["O"] * n_tokens
    for start, end, kind in spans:
        labels[start] = "B-" + kind
        for i in range(start + 1, end):
            labels[i] = "I-" + kind
    return labels


def spans_to_bilou(n_tokens, spans):
    labels = ["O"] * n_tokens
    for start, end, kind in spans:
        if end - start == 1:
            labels[start] = "U-" + kind
        else:
            labels[start] = "B-" + kind
            for i in range(start + 1, end - 1):
                labels[i] = "I-" + kind
            labels[end - 1] = "L-" + kind
    return labels


def bilou_to_spans(labels):
    spans, current = [], None
    for i, label in enumerate(labels):
        prefix, _, kind = label.partition("-")
        if prefix == "U":
            if current:
                spans.append(current)
                current = None
            spans.append((i, i + 1, kind))
        elif prefix == "B":
            if current:
                spans.append(current)
            current = (i, i + 1, kind)
        elif prefix == "I" and current and current[2] == kind:
            current = (current[0], i + 1, kind)
        elif prefix == "L" and current and current[2] == kind:
            spans.append((current[0], i + 1, kind))
            current = None
        else:
            if current:
                spans.append(current)
                current = None
    if current:
        spans.append(current)
    return spans


tokens = ["Bank", "of", "America", "Tower", "in", "New", "York", "City", "."]
org_span = (0, 3, "ORG")
facility_span = (0, 4, "FAC")

single_pass = spans_to_bio(len(tokens), [org_span, facility_span])
print("--- một mảng nhãn, hai span chồng lấn ---")
print("tokens     :", tokens)
print("BIO        :", single_pass)
print("thu hồi    :", bio_to_spans(single_pass))
print("ORG bị mất vì span FAC ghi đè lên cả bốn token.")

print("\n--- multi-pass: mỗi kiểu một mảng nhãn ---")
org_pass = spans_to_bio(len(tokens), [org_span])
facility_pass = spans_to_bio(len(tokens), [facility_span])
print("lượt ORG   :", org_pass)
print("lượt FAC   :", facility_pass)
print("thu hồi ORG:", bio_to_spans(org_pass))
print("thu hồi FAC:", bio_to_spans(facility_pass))
print("Hai lượt giữ được cả hai span vì mỗi lượt là bài toán nhị phân độc lập.")

print("\n--- BILOU làm rõ ranh giới ---")
spans = [(1, 4, "GPE"), (5, 6, "GPE"), (7, 8, "ORG")]
tokens2 = ["The", "New", "York", "City", "mayor", "visited", "Acme", "."]
bio = spans_to_bio(len(tokens2), spans)
bilou = spans_to_bilou(len(tokens2), spans)
print("tokens     :", tokens2)
print("BIO        :", bio)
print("BILOU      :", bilou)
print("round-trip :", bilou_to_spans(bilou))
print("khớp gold  :", bilou_to_spans(bilou) == [s for s in spans])
# U-GPE cho thực thể một token, L-GPE cho token cuối của 'New York City'.


--- một mảng nhãn, hai span chồng lấn ---
tokens     : ['Bank', 'of', 'America', 'Tower', 'in', 'New', 'York', 'City', '.']
BIO        : ['B-FAC', 'I-FAC', 'I-FAC', 'I-FAC', 'O', 'O', 'O', 'O', 'O']
thu hồi    : [(0, 4, 'FAC')]
ORG bị mất vì span FAC ghi đè lên cả bốn token.

--- multi-pass: mỗi kiểu một mảng nhãn ---
lượt ORG   : ['B-ORG', 'I-ORG', 'I-ORG', 'O', 'O', 'O', 'O', 'O', 'O']
lượt FAC   : ['B-FAC', 'I-FAC', 'I-FAC', 'I-FAC', 'O', 'O', 'O', 'O', 'O']
thu hồi ORG: [(0, 3, 'ORG')]
thu hồi FAC: [(0, 4, 'FAC')]
Hai lượt giữ được cả hai span vì mỗi lượt là bài toán nhị phân độc lập.

--- BILOU làm rõ ranh giới ---
tokens     : ['The', 'New', 'York', 'City', 'mayor', 'visited', 'Acme', '.']
BIO        : ['O', 'B-GPE', 'I-GPE', 'I-GPE', 'O', 'B-GPE', 'O', 'B-ORG']
BILOU      : ['O', 'B-GPE', 'I-GPE', 'L-GPE', 'O', 'U-GPE', 'O', 'U-ORG']
round-trip : [(1, 4, 'GPE'), (5, 6, 'GPE'), (7, 8, 'ORG')]
khớp gold  : True


## D. Tự kiểm tra

1. Vì sao `O I-ORG` là chuỗi bất hợp lệ, và CRF ngăn nó bằng cách nào?
2. Token accuracy 0.92 có thể đi kèm entity F1 thấp. Giải thích cơ chế.
3. BILOU thêm thông tin gì mà BIO thiếu?
4. Vì sao gazetteer không phân biệt được `Apple` công ty và quả táo?
5. Nested entity cần kiến trúc gì, và vì sao BIO không đủ?
6. Khi nào nên chọn LLM zero-shot thay vì CRF?

**Giới hạn:** gazetteer và bộ luật trong notebook rất nhỏ, được thiết kế để có lỗi quan sát được. Điểm phát xạ ở phần B do người viết đặt, không phải đầu ra mô hình đã huấn luyện. Kết quả chỉ minh họa cơ chế gán nhãn, giải mã và đo lường.
